# Monte Carlo vs Black-Scholes Comprehensive Comparison

This notebook provides a comprehensive comparison of Monte Carlo simulation against Black-Scholes analytical solutions.

## Topics Covered:
1. Price Accuracy Across Moneyness
2. Greeks Accuracy Comparison
3. Computational Performance
4. Asian Options (No Closed Form)
5. Method Selection Guidelines

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import sys
sys.path.insert(0, '../src')

from mc_pricing import (
    EuropeanOption, AsianOption, BlackScholes,
    MonteCarloSimulator, Visualizer
)
from mc_pricing.greeks import FiniteDifferenceGreeks
from mc_pricing.variance_reduction import AntitheticVariates, ControlVariates
from mc_pricing.utils import PerformanceMetrics

%matplotlib inline
plt.style.use('seaborn-v0_8')

## 1. Price Accuracy Across Moneyness Spectrum

In [ ]:
# Test across different moneyness levels
spot = 100.0
maturity = 1.0
rate = 0.05
volatility = 0.2

strikes = np.array([70, 80, 85, 90, 95, 100, 105, 110, 115, 120, 130])
n_simulations = 100000

print("Pricing Comparison Across Strikes (Call Options)")
print(f"Using {n_simulations:,} simulations with Control Variates\n")

results = []

for strike in strikes:
    # Black-Scholes price
    bs_price = BlackScholes.price('call', spot, strike, maturity, rate, volatility)
    
    # Monte Carlo price
    option = EuropeanOption('call', strike, maturity, spot, rate, volatility)
    simulator = MonteCarloSimulator(n_simulations=n_simulations, seed=42)
    mc_price, mc_error = simulator.price(option, variance_reduction=ControlVariates())
    
    moneyness = strike / spot
    abs_error = abs(mc_price - bs_price)
    rel_error = (abs_error / bs_price * 100) if bs_price > 0 else 0
    
    results.append({
        'Strike': strike,
        'Moneyness': moneyness,
        'BS_Price': bs_price,
        'MC_Price': mc_price,
        'MC_StdErr': mc_error,
        'Abs_Error': abs_error,
        'Rel_Error_%': rel_error
    })

df = pd.DataFrame(results)
print(df[['Strike', 'Moneyness', 'BS_Price', 'MC_Price', 'MC_StdErr', 'Abs_Error', 'Rel_Error_%']].to_string(index=False))

print(f"\nAverage Relative Error: {df['Rel_Error_%'].mean():.3f}%")
print(f"Max Relative Error: {df['Rel_Error_%'].max():.3f}%")

In [ ]:
# Visualize price comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Price comparison
ax1.plot(df['Moneyness'], df['BS_Price'], 'o-', label='Black-Scholes', 
         markersize=8, linewidth=2.5, color='blue')
ax1.errorbar(df['Moneyness'], df['MC_Price'], yerr=2*df['MC_StdErr'], 
             fmt='s--', label='Monte Carlo (95% CI)', 
             markersize=7, linewidth=2, color='red', capsize=5)
ax1.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5, label='ATM')
ax1.set_xlabel('Moneyness (K/S)', fontsize=12)
ax1.set_ylabel('Option Price ($)', fontsize=12)
ax1.set_title('Price Comparison', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Error analysis
ax2.bar(df['Moneyness'], df['Rel_Error_%'], alpha=0.7, color='coral', edgecolor='black')
ax2.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Moneyness (K/S)', fontsize=12)
ax2.set_ylabel('Relative Error (%)', fontsize=12)
ax2.set_title('Monte Carlo Relative Error', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. Greeks Comparison

In [ ]:
# Compare Greeks for ATM option
atm_option = EuropeanOption('call', 100, 1.0, 100, 0.05, 0.2)

# Black-Scholes Greeks
bs_greeks = BlackScholes.greeks('call', 100, 100, 1.0, 0.05, 0.2)

# Monte Carlo Greeks
simulator = MonteCarloSimulator(n_simulations=100000, seed=42)
fd_greeks = FiniteDifferenceGreeks(simulator)

mc_greeks = {
    'delta': fd_greeks.delta(atm_option),
    'gamma': fd_greeks.gamma(atm_option),
    'vega': fd_greeks.vega(atm_option),
    'theta': fd_greeks.theta(atm_option),
    'rho': fd_greeks.rho(atm_option)
}

# Comparison table
greek_comparison = pd.DataFrame({
    'Greek': ['Delta', 'Gamma', 'Vega', 'Theta', 'Rho'],
    'Black-Scholes': [
        bs_greeks['delta'],
        bs_greeks['gamma'],
        bs_greeks['vega'],
        bs_greeks['theta'],
        bs_greeks['rho']
    ],
    'Monte Carlo': [
        mc_greeks['delta'],
        mc_greeks['gamma'],
        mc_greeks['vega'],
        mc_greeks['theta'],
        mc_greeks['rho']
    ]
})

greek_comparison['Absolute Error'] = abs(
    greek_comparison['Black-Scholes'] - greek_comparison['Monte Carlo']
)
greek_comparison['Relative Error %'] = (
    greek_comparison['Absolute Error'] / abs(greek_comparison['Black-Scholes']) * 100
)

print("Greeks Comparison (ATM Call Option):")
print(greek_comparison.to_string(index=False))
print(f"\nAverage Relative Error: {greek_comparison['Relative Error %'].mean():.2f}%")

In [ ]:
# Visualize Greeks comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(greek_comparison))
width = 0.35

bars1 = ax.bar(x - width/2, greek_comparison['Black-Scholes'], width, 
               label='Black-Scholes', alpha=0.8, color='blue')
bars2 = ax.bar(x + width/2, greek_comparison['Monte Carlo'], width, 
               label='Monte Carlo', alpha=0.8, color='red')

ax.set_xlabel('Greek', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('Greeks Comparison: Black-Scholes vs Monte Carlo', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(greek_comparison['Greek'])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 3. Computational Performance Benchmark

In [ ]:
# Benchmark pricing speed
option = EuropeanOption('call', 100, 1.0, 100, 0.05, 0.2)
n_runs = 50

print("Performance Benchmark:")
print(f"Running {n_runs} independent trials...\n")

# Black-Scholes
bs_times = []
for _ in range(n_runs):
    start = time.perf_counter()
    BlackScholes.price('call', 100, 100, 1.0, 0.05, 0.2)
    bs_times.append(time.perf_counter() - start)

# Monte Carlo (various simulation counts)
mc_configs = [
    ('MC 10k', 10000),
    ('MC 50k', 50000),
    ('MC 100k', 100000),
    ('MC 100k + CV', 100000)
]

performance_results = [{
    'Method': 'Black-Scholes',
    'Mean Time (ms)': np.mean(bs_times) * 1000,
    'Std Time (ms)': np.std(bs_times) * 1000,
    'Speedup': 1.0
}]

for name, n_sims in mc_configs:
    mc_times = []
    for i in range(n_runs):
        simulator = MonteCarloSimulator(n_simulations=n_sims, seed=42+i)
        start = time.perf_counter()
        if 'CV' in name:
            simulator.price(option, variance_reduction=ControlVariates())
        else:
            simulator.price(option)
        mc_times.append(time.perf_counter() - start)
    
    mean_time = np.mean(mc_times)
    performance_results.append({
        'Method': name,
        'Mean Time (ms)': mean_time * 1000,
        'Std Time (ms)': np.std(mc_times) * 1000,
        'Speedup': np.mean(bs_times) / mean_time
    })

perf_df = pd.DataFrame(performance_results)
print(perf_df.to_string(index=False))
print(f"\nBlack-Scholes is {perf_df.loc[perf_df['Method']=='MC 100k', 'Speedup'].values[0]:.0f}x faster than MC (100k sims)")

In [ ]:
# Visualize performance
fig, ax = plt.subplots(figsize=(10, 6))

methods = perf_df['Method']
times = perf_df['Mean Time (ms)']
errors = perf_df['Std Time (ms)']

colors = ['blue', 'coral', 'coral', 'coral', 'green']
bars = ax.bar(methods, times, yerr=errors, capsize=5, alpha=0.7, 
              color=colors, edgecolor='black')

ax.set_ylabel('Computation Time (ms)', fontsize=12)
ax.set_title('Computational Performance Comparison', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=45)

# Add speedup annotations
for i, (method, speedup) in enumerate(zip(methods, perf_df['Speedup'])):
    if speedup < 1:
        ax.text(i, times.iloc[i], f'{1/speedup:.0f}x slower', 
               ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Asian Options - Monte Carlo Advantage

No closed-form solution exists for arithmetic Asian options!

In [ ]:
# Price Asian options with different averaging types
print("Asian Options Pricing (Monte Carlo Only):")
print(f"European options have closed-form, but Asian options don't!\n")

simulator = MonteCarloSimulator(n_simulations=100000, seed=42)

# European for comparison
european_call = EuropeanOption('call', 100, 1.0, 100, 0.05, 0.2)
euro_price, euro_err = simulator.price(european_call, variance_reduction=AntitheticVariates())
euro_bs = BlackScholes.price('call', 100, 100, 1.0, 0.05, 0.2)

# Arithmetic Asian
asian_arith = AsianOption('call', 100, 1.0, 100, 0.05, 0.2, averaging_type='arithmetic')
asian_arith_price, asian_arith_err = simulator.price(asian_arith, variance_reduction=AntitheticVariates())

# Geometric Asian
asian_geom = AsianOption('call', 100, 1.0, 100, 0.05, 0.2, averaging_type='geometric')
asian_geom_price, asian_geom_err = simulator.price(asian_geom, variance_reduction=AntitheticVariates())

asian_comparison = pd.DataFrame({
    'Option Type': ['European', 'Asian (Arithmetic)', 'Asian (Geometric)'],
    'MC Price': [euro_price, asian_arith_price, asian_geom_price],
    'Std Error': [euro_err, asian_arith_err, asian_geom_err],
    'BS Price': [euro_bs, 'N/A', 'N/A'],
    'Has Closed Form': ['Yes', 'No', 'Semi-closed']
})

print(asian_comparison.to_string(index=False))

print("\nKey Insight:")
print(f"  Asian options are cheaper than European ({asian_arith_price:.4f} < {euro_price:.4f})")
print(f"  Geometric average < Arithmetic average ({asian_geom_price:.4f} < {asian_arith_price:.4f})")
print(f"\n  For arithmetic Asian options, Monte Carlo is ESSENTIAL!")

## 5. Method Selection Guidelines

In [ ]:
# Create decision matrix
decision_matrix = pd.DataFrame({
    'Scenario': [
        'European option, need speed',
        'European option, need Greeks',
        'Asian option (arithmetic)',
        'Asian option (geometric)',
        'Exotic path-dependent',
        'American option',
        'Multi-asset option',
        'Research/validation',
        'Production pricing',
        'Risk management'
    ],
    'Recommended Method': [
        'Black-Scholes',
        'Black-Scholes',
        'Monte Carlo (required)',
        'Closed-form or MC',
        'Monte Carlo (required)',
        'Binomial/FD',
        'Monte Carlo',
        'Both (validation)',
        'MC with variance reduction',
        'Black-Scholes + MC'
    ],
    'Reason': [
        'Fastest, exact',
        'Closed-form Greeks',
        'No closed form',
        'Has semi-closed form',
        'Only viable method',
        'Better than MC for early exercise',
        'Handles correlations',
        'Cross-validate methods',
        'Good accuracy-speed trade-off',
        'Use both for robustness'
    ]
})

print("\nMethod Selection Guide:")
print("=" * 100)
print(decision_matrix.to_string(index=False))
print("=" * 100)

## 6. Accuracy vs Speed Trade-off

In [ ]:
# Test different simulation counts
option = EuropeanOption('call', 100, 1.0, 100, 0.05, 0.2)
bs_price = BlackScholes.price('call', 100, 100, 1.0, 0.05, 0.2)

sim_counts = [1000, 5000, 10000, 50000, 100000, 500000]
tradeoff_results = []

for n_sims in sim_counts:
    # Time and price
    simulator = MonteCarloSimulator(n_simulations=n_sims, seed=42)
    
    start = time.perf_counter()
    mc_price, mc_error = simulator.price(option, variance_reduction=ControlVariates())
    elapsed = time.perf_counter() - start
    
    accuracy = abs(mc_price - bs_price)
    
    tradeoff_results.append({
        'Simulations': n_sims,
        'Time (ms)': elapsed * 1000,
        'Accuracy ($)': accuracy,
        'Std Error ($)': mc_error
    })

tradeoff_df = pd.DataFrame(tradeoff_results)

# Plot trade-off
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(tradeoff_df['Time (ms)'], tradeoff_df['Accuracy ($)'], 
          s=200, c=np.log10(tradeoff_df['Simulations']), 
          cmap='viridis', edgecolor='black', linewidth=2)

# Add labels
for _, row in tradeoff_df.iterrows():
    ax.annotate(f"{row['Simulations']:,}", 
               (row['Time (ms)'], row['Accuracy ($)']),
               xytext=(5, 5), textcoords='offset points', fontsize=9)

ax.set_xlabel('Computation Time (ms)', fontsize=12)
ax.set_ylabel('Absolute Error vs BS ($)', fontsize=12)
ax.set_title('Accuracy vs Speed Trade-off (with Control Variates)', 
            fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Add optimal region
ax.axhspan(0, 0.05, alpha=0.1, color='green', label='High Accuracy Zone')
ax.axvspan(0, 100, alpha=0.1, color='blue', label='Fast Zone')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\nSweet Spot: 50k-100k simulations with variance reduction")
print("  - Good accuracy (<$0.05 error)")
print("  - Reasonable speed (<100ms)")
print("  - Production-ready performance")

## Summary and Final Recommendations

### Performance Summary:

| Metric | Black-Scholes | Monte Carlo (100k + CV) |
|--------|---------------|-------------------------|
| Speed | ~0.01 ms | ~100 ms |
| Accuracy | Exact | ±0.05 (95% CI) |
| European Options | ✅ Best | ✅ Good |
| Asian Options | ❌ No | ✅ Required |
| Greeks | ✅ Instant | ⚠️ Slower |
| Path-dependent | ❌ Limited | ✅ Full support |

### Key Takeaways:

1. **Black-Scholes**: 
   - Use for European options when speed is critical
   - Perfect for Greeks calculation
   - Limited to simple payoffs

2. **Monte Carlo**:
   - Required for path-dependent options (Asian, Lookback, etc.)
   - Flexible for any payoff structure
   - 100k-500k simulations recommended for production
   - Always use variance reduction

3. **Hybrid Approach**:
   - Use BS for European pricing and Greeks
   - Use MC for exotic and path-dependent options
   - Cross-validate MC against BS for European options

### Production Setup:
```python
# Recommended configuration
simulator = MonteCarloSimulator(
    n_simulations=100000,  # Good balance
    seed=42                # Reproducibility
)

# Use control variates for best efficiency
price, error = simulator.price(
    option,
    variance_reduction=ControlVariates()
)
```

### When to Use Each Method:

- **Black-Scholes**: Standard European options, risk management, quick quotes
- **Monte Carlo**: Exotics, path-dependent, multi-asset, research
- **Both**: Production systems (BS for speed, MC for validation)

## Conclusion

Monte Carlo is a powerful, flexible method that complements analytical solutions. 
While slower than Black-Scholes for European options, it's often the **only** 
viable method for complex derivatives. With variance reduction techniques, 
Monte Carlo achieves production-grade accuracy and performance.

---

**End of Notebook Series**

You've now covered:
1. Basic pricing ✅
2. Variance reduction ✅
3. Greeks analysis ✅
4. Convergence study ✅
5. Smile calibration ✅
6. Full comparison ✅

Happy option pricing! 🎯📈